In [32]:
import torch
from tensorflow.keras.preprocessing.text import Tokenizer
from google.colab import drive
drive.mount("/content/drive")
ckpt = torch.load("/content/drive/MyDrive/model/sentiment_inference.pth", weights_only=False)
word_index = ckpt['tokenizer']
tokenizer = Tokenizer(num_words=ckpt['vocab_size'], oov_token='<OOV>')
tokenizer.word_index = word_index
import torch
import torch.nn as nn
import torch.nn.functional as F

class SentimentModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(SentimentModel,self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 3)

    def forward(self, x):
        x = self.embedding(x)
        x = x.mean(dim=1)
        x = F.relu(self.fc1(x)) # final hidden state
        x = self.fc2(x)
        return x


model = SentimentModel(
    ckpt['vocab_size'],
    ckpt['embed_dim'],
    ckpt['hidden_dim']
)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print("model loaded successfully!")
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict(text):
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=ckpt['max_len'], padding='post')
    X_tensor = torch.tensor(pad, dtype=torch.long)
    with torch.no_grad():
        output = model(X_tensor)
        sentiment = torch.argmax(output, dim=1).item()
        return sentiment

predict("aku suka ini")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
model loaded successfully!


2